# modelgeometry — live demo (Colab, T4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Siddhesh290307/modelgeometry/blob/main/examples/colab_demo.ipynb)

This notebook exercises the **entire [`modelgeometry`](https://pypi.org/project/modelgeometry/) public API** against real **pretrained** checkpoints — a real GPT-2 (`gpt2`, from HuggingFace) and a real pretrained Vision Transformer (`vit_base_patch16_224`, from `timm`). Nothing is trained from scratch; the only "training" here is ~40 lightweight finetuning steps on a handful of sentences, purely to give `GeometryTracker` and `compare_checkpoints` something to show.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`.

Sections:

1. Install + load pretrained models
2. Weight-space geometry (`modelgeometry.linalg`)
3. Attention geometry (`modelgeometry.attention`)
4. Remaining weight/activation primitives
5. Diagonal Fisher information (`modelgeometry.fisher`)
6. K-FAC curvature factors (`modelgeometry.kfac`)
7. Curvature-prediction check (`modelgeometry.curvature`)
8. Regularizers (`modelgeometry.regularizers`)
9. Light finetuning + `GeometryTracker` + `compare_checkpoints`
10. Cross-architecture check: the same functions on a pretrained ViT
11. Save + download all figures

Every figure is saved as a PNG so it can be pulled into the project README afterward.


## 1. Install + load pretrained models

No training from scratch anywhere in this notebook — both models below are loaded with their published pretrained weights.

In [ ]:
!pip install -q modelgeometry[report] transformers timm


In [ ]:
import copy
import re
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import timm
import torch
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer

from modelgeometry import (
    EWCPenalty,
    GeometryTracker,
    HookRegistry,
    KFACPenalty,
    L2Penalty,
    SynapticIntelligencePenalty,
    attention_effective_rank,
    attention_entropy,
    capture_attention_weights,
    compare_checkpoints,
    curvature_prediction_check,
    diagonal_fisher,
    distributional_distance,
    effective_rank,
    fisher_layer_summary,
    frobenius_norm,
    kfac_factors,
    kfac_offdiagonal_energy,
    nullspace_projection,
    participation_ratio,
    qkv_norm_stats,
    resolve_adapter,
    row_cosine_similarity,
    spectral_norm,
)
from modelgeometry.report import plot_checkpoint_comparison, plot_tracker_history

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


In [ ]:
# --- GPT-2 (pretrained, eager attention so we can capture attention weights
# and so torch.func's per-sample-gradient machinery avoids fused-kernel
# batching-rule gaps) ---
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained("gpt2", attn_implementation="eager")
model.to(device)
model.eval()

# --- ViT (pretrained), used later to show the same functions work unmodified
# on a completely different architecture ---
vit = timm.create_model("vit_base_patch16_224", pretrained=True)
vit.to(device)
vit.eval()

print(f"GPT-2: {model.config.n_layer} layers, {model.config.n_head} heads, {model.config.n_embd} hidden")
print(f"ViT:   {sum(p.numel() for p in vit.parameters()) / 1e6:.1f}M params")


In [ ]:
# A small, fixed set of sentences used throughout as "real" input data —
# just enough for meaningful activations/gradients, not a training corpus.
texts = [
    "The quick brown fox jumps over the lazy dog.",
    "Climate models predict rising sea levels over the next century.",
    "The stock market fluctuated wildly after the announcement.",
    "She carefully tuned the guitar before the evening performance.",
    "Researchers discovered a new species of deep-sea fish.",
    "The recipe calls for two cups of flour and a pinch of salt.",
    "Quantum computers exploit superposition to solve certain problems faster.",
    "The marathon route winds through the historic old town.",
]

encoded = tokenizer(texts, return_tensors="pt", padding="max_length", truncation=True, max_length=32)
input_ids = encoded["input_ids"].to(device)


def lm_loss_fn(predictions, target):
    """Standard shifted causal-LM cross-entropy, padding ignored."""
    logits = predictions.logits if hasattr(predictions, "logits") else predictions
    logits = logits[:, :-1, :].contiguous()
    labels = target[:, 1:].contiguous()
    return F.cross_entropy(logits.reshape(-1, logits.shape[-1]), labels.reshape(-1), ignore_index=tokenizer.pad_token_id)


def make_dataloader(n_batches, batch_size):
    batches = []
    for i in range(n_batches):
        rows = input_ids[(i * batch_size) % len(texts) : (i * batch_size) % len(texts) + batch_size]
        if rows.shape[0] < batch_size:
            rows = input_ids[:batch_size]
        batches.append((rows, rows))
    return batches


adapter_gpt2 = resolve_adapter(model)
print("adapter:", type(adapter_gpt2).__name__, "| layers:", adapter_gpt2.num_layers())


## 2. Weight-space geometry

No forward pass needed for this section — these read straight off the pretrained weights.

In [ ]:
ranks = [effective_rank(adapter_gpt2.qkv_weights(i).q) for i in range(adapter_gpt2.num_layers())]
norms = [spectral_norm(adapter_gpt2.qkv_weights(i).q) for i in range(adapter_gpt2.num_layers())]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(ranks, marker="o")
axes[0].set_title("GPT-2: Q-projection effective rank per layer")
axes[0].set_xlabel("layer")
axes[0].set_ylabel("effective rank")
axes[1].plot(norms, marker="o", color="darkorange")
axes[1].set_title("GPT-2: Q-projection spectral norm per layer")
axes[1].set_xlabel("layer")
axes[1].set_ylabel("spectral norm")
plt.tight_layout()
plt.savefig("gpt2_weight_geometry.png", dpi=150)
plt.show()


In [ ]:
sim = row_cosine_similarity(adapter_gpt2.qkv_weights(0).q)

plt.figure(figsize=(5, 5))
plt.imshow(sim, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="cosine similarity")
plt.title("GPT-2 layer 0: Q-projection row cosine similarity")
plt.tight_layout()
plt.savefig("gpt2_row_cosine_similarity.png", dpi=150)
plt.show()


## 3. Attention geometry

Real attention weights, captured via `HookRegistry` on a batch of real sentences.

In [ ]:
entropy_grid = np.zeros((adapter_gpt2.num_layers(), model.config.n_head))
rank_grid = np.zeros_like(entropy_grid)

for layer_idx in range(adapter_gpt2.num_layers()):
    registry = HookRegistry()
    with registry:
        capture_attention_weights(registry, "attn", adapter_gpt2.attention_module(layer_idx))
        with torch.no_grad():
            model(input_ids, output_attentions=True)
    weights = registry.captured["attn"]  # (batch, heads, seq, seq)
    entropy_grid[layer_idx] = attention_entropy(weights).mean(axis=(0, 2))
    rank_grid[layer_idx] = attention_effective_rank(weights).mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im0 = axes[0].imshow(entropy_grid, cmap="viridis", aspect="auto")
axes[0].set_title("Attention entropy (layer x head)")
axes[0].set_xlabel("head")
axes[0].set_ylabel("layer")
plt.colorbar(im0, ax=axes[0])
im1 = axes[1].imshow(rank_grid, cmap="viridis", aspect="auto")
axes[1].set_title("Attention effective rank (layer x head)")
axes[1].set_xlabel("head")
axes[1].set_ylabel("layer")
plt.colorbar(im1, ax=axes[1])
plt.tight_layout()
plt.savefig("gpt2_attention_heatmaps.png", dpi=150)
plt.show()


## 4. Remaining weight/activation primitives

Quick round-up of the primitives not already covered above.

In [ ]:
registry = HookRegistry()
with registry:
    registry.capture_output("fused_qkv_out", adapter_gpt2.qkv_modules(0)["qkv"])
    with torch.no_grad():
        model(input_ids)
q_act, k_act, v_act = registry.captured["fused_qkv_out"].split(model.config.n_embd, dim=-1)
print("qkv_norm_stats:", qkv_norm_stats(q_act, k_act, v_act))

mass_values = np.array([spectral_norm(adapter_gpt2.qkv_weights(i).q) for i in range(adapter_gpt2.num_layers())])
print("participation_ratio of per-layer spectral norms:", participation_ratio(mass_values))

d = distributional_distance(adapter_gpt2.qkv_weights(0).q.flatten(), adapter_gpt2.qkv_weights(-1).q.flatten())
print("distributional_distance, layer 0 vs last-layer Q weights:", d)

print("frobenius_norm, layer 0 Q:", frobenius_norm(adapter_gpt2.qkv_weights(0).q))
print("nullspace_projection shape:", nullspace_projection(model.config.n_embd).shape)


## 5. Diagonal Fisher information

Empirical diagonal Fisher via true per-sample gradients (Kirkpatrick et al., 2017), computed on the small real-text dataloader.

In [ ]:
dataloader = make_dataloader(n_batches=4, batch_size=4)
fisher = diagonal_fisher(model, dataloader, n_samples=16, loss_fn=lm_loss_fn)
summary = fisher_layer_summary(fisher, top_k=10)

print("total Fisher mass:", summary["total_mass"])
print("top-10 mass fraction:", summary["top_k_mass_fraction"])
print("effective rank of the per-parameter mass distribution:", summary["effective_rank"])

per_layer_mass = {}
for name, mass in summary["per_parameter_mass"].items():
    match = re.search(r"h\.(\d+)\.", name)
    layer = int(match.group(1)) if match else -1
    per_layer_mass[layer] = per_layer_mass.get(layer, 0.0) + mass

layers_sorted = sorted(k for k in per_layer_mass if k >= 0)
plt.figure(figsize=(8, 4))
plt.bar(layers_sorted, [per_layer_mass[l] for l in layers_sorted])
plt.xlabel("layer")
plt.ylabel("summed diagonal Fisher mass")
plt.title("GPT-2: diagonal Fisher mass per layer")
plt.tight_layout()
plt.savefig("gpt2_fisher_mass_per_layer.png", dpi=150)
plt.show()


## 6. K-FAC curvature factors

Martens & Grosse, 2015 — activation- and gradient-covariance factors per attention projection.

In [ ]:
factors = kfac_factors(model, dataloader, n_samples=16, loss_fn=lm_loss_fn, adapter=adapter_gpt2, include_bias=False)
offdiag = kfac_offdiagonal_energy(factors)

names = list(offdiag.keys())
values = [offdiag[n] for n in names]
labels = [n.replace("transformer.h.", "L").replace(".attn.c_attn.weight", "") for n in names]

plt.figure(figsize=(10, 4))
plt.bar(range(len(names)), values)
plt.xticks(range(len(names)), labels, rotation=45, ha="right")
plt.ylabel("off-diagonal energy fraction")
plt.title("GPT-2: K-FAC off-diagonal energy per layer's fused QKV projection")
plt.tight_layout()
plt.savefig("gpt2_kfac_offdiagonal_energy.png", dpi=150)
plt.show()


## 7. Curvature-prediction check

Compares the actual loss change under a small random perturbation to the diagonal-Fisher-predicted change.

In [ ]:
perturbations = {name: 1e-3 * torch.randn_like(p) for name, p in model.named_parameters()}
result = curvature_prediction_check(
    model,
    loss_fn=lm_loss_fn,
    batch=(dataloader[0][0], dataloader[0][0]),
    perturbations=perturbations,
    fisher=fisher,
)
for key, value in result.items():
    print(f"{key}: {value}")


## 8. Regularizers

Run against a **throwaway deep copy** of the pretrained model, so nothing here disturbs the pretrained weights used later for the finetuning comparison.

In [ ]:
demo_model = copy.deepcopy(model).to(device)
demo_model.eval()
anchor = {name: p.detach().clone() for name, p in demo_model.named_parameters()}

l2 = L2Penalty(demo_model, weight=1e-4)
print("L2 penalty:", l2.penalty().item())

ewc = EWCPenalty(demo_model, fisher=fisher, anchor_params=anchor, weight=1.0)
print("EWC penalty at anchor:", ewc.penalty().item())
with torch.no_grad():
    demo_model.transformer.h[0].attn.c_attn.weight.add_(0.01 * torch.randn_like(demo_model.transformer.h[0].attn.c_attn.weight))
print("EWC penalty after perturbing layer 0:", ewc.penalty().item())
with torch.no_grad():
    demo_model.transformer.h[0].attn.c_attn.weight.copy_(anchor["transformer.h.0.attn.c_attn.weight"])

si = SynapticIntelligencePenalty(demo_model)
optimizer = torch.optim.SGD(demo_model.parameters(), lr=1e-4)
demo_model.train()
for _ in range(3):
    optimizer.zero_grad()
    loss = lm_loss_fn(demo_model(input_ids), input_ids)
    loss.backward()
    optimizer.step()
    si.step()
si.consolidate()
demo_model.eval()
print("Synaptic Intelligence penalty right after consolidate (should be ~0):", si.penalty().item())

kfac_reg = KFACPenalty(demo_model, kfac_factors=factors, anchor_params=anchor, weight=1.0)
print("K-FAC penalty vs original anchor (post SI drift):", kfac_reg.penalty().item())


## 9. Light finetuning + `GeometryTracker` + `compare_checkpoints`

The **real, untouched** pretrained `model` from section 1 — a short (~40 step) finetune on the same small sentence set, tracked step by step.

In [ ]:
pretrained_snapshot = copy.deepcopy(model).eval()


def qkv0_effective_rank(m, a):
    return effective_rank(a.qkv_weights(0).q)


def layer0_attn_entropy(m, a):
    registry = HookRegistry()
    with registry:
        capture_attention_weights(registry, "attn", a.attention_module(0))
        with torch.no_grad():
            m(input_ids, output_attentions=True)
    return float(attention_entropy(registry.captured["attn"]).mean())


tracker = GeometryTracker(model, metrics=[("qkv0_effective_rank", qkv0_effective_rank), ("layer0_attn_entropy", layer0_attn_entropy)])

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
model.train()
for step in range(40):
    optimizer.zero_grad()
    loss = lm_loss_fn(model(input_ids), input_ids)
    loss.backward()
    optimizer.step()
    if step % 5 == 0:
        model.eval()
        tracker.log_step(step)
        model.train()
model.eval()

print(f"Logged {len(tracker.history)} steps; final loss: {loss.item():.4f}")


In [ ]:
ax = plot_tracker_history(tracker.history, "qkv0_effective_rank")
ax.figure.tight_layout()
ax.figure.savefig("gpt2_training_qkv_rank.png", dpi=150)
plt.show()

ax = plot_tracker_history(tracker.history, "layer0_attn_entropy")
ax.figure.tight_layout()
ax.figure.savefig("gpt2_training_attn_entropy.png", dpi=150)
plt.show()


In [ ]:
report = compare_checkpoints(
    pretrained_snapshot,
    model,
    metrics=[("qkv0_effective_rank", qkv0_effective_rank), ("layer0_attn_entropy", layer0_attn_entropy)],
)
for name, entry in report.items():
    print(name, entry)

ax = plot_checkpoint_comparison(report)
ax.figure.tight_layout()
ax.figure.savefig("gpt2_pretrained_vs_finetuned.png", dpi=150)
plt.show()


## 10. Cross-architecture check: the same functions on a pretrained ViT

No attention-specific special-casing anywhere below — `resolve_adapter` and `effective_rank` are the exact same calls used on GPT-2 above.

In [ ]:
adapter_vit = resolve_adapter(vit)
print("adapter:", type(adapter_vit).__name__, "| layers:", adapter_vit.num_layers())

vit_ranks = [effective_rank(adapter_vit.qkv_weights(i).q) for i in range(adapter_vit.num_layers())]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ranks, marker="o", label="GPT-2 (fused QKV, Conv1D)")
ax.plot(vit_ranks, marker="s", label="ViT (fused QKV, nn.Linear)")
ax.set_xlabel("layer")
ax.set_ylabel("Q-projection effective rank")
ax.set_title("Same modelgeometry.effective_rank() call across two architectures")
ax.legend()
plt.tight_layout()
plt.savefig("cross_architecture_effective_rank.png", dpi=150)
plt.show()


## 11. Save + download all figures

In [ ]:
import glob

png_files = sorted(glob.glob("*.png"))
print("Saved figures:")
for f in png_files:
    print(" -", f)

with zipfile.ZipFile("modelgeometry_demo_figures.zip", "w") as zf:
    for f in png_files:
        zf.write(f)

try:
    from google.colab import files

    files.download("modelgeometry_demo_figures.zip")
except ImportError:
    print("Not running in Colab — figures are saved locally in the working directory.")


---

That's the full `modelgeometry` public API exercised against two real pretrained architectures. Pick whichever figures are most illustrative from `modelgeometry_demo_figures.zip` and share them back — they're ready to drop into the project README.